In [ ]:
# =========================
# Cell 1. Install / imports
# =========================

!pip -q install requests pandas scikit-learn tqdm matplotlib

import re
import time
import math
import json
import warnings
from collections import Counter

import requests
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")


# =========================
# Cell 2. Config
# =========================

BASE_URL = "https://public-api.prozorro.gov.ua/api/2.5"

FEED_LIMIT = 100
MAX_FEED_ITEMS = 2500
REQUEST_TIMEOUT = 30
SLEEP_BETWEEN_REQUESTS = 0.0

RANDOM_STATE = 42

session = requests.Session()
session.headers.update({
    "Accept": "application/json",
    "User-Agent": "Mozilla/5.0"
})


# =========================
# Cell 3. API helpers
# =========================

def fetch_feed_page(offset=None, limit=100):
    params = {"limit": limit}
    if offset:
        params["offset"] = offset

    r = session.get(f"{BASE_URL}/tenders", params=params, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.json()


def collect_feed_items(max_items=500):
    items = []
    offset = None

    while len(items) < max_items:
        batch_limit = min(FEED_LIMIT, max_items - len(items))
        data = fetch_feed_page(offset=offset, limit=batch_limit)

        batch = data.get("data") or []
        if not batch:
            break

        items.extend(batch)

        next_page = data.get("next_page") or {}
        offset = next_page.get("offset")

        if not offset:
            break

        if SLEEP_BETWEEN_REQUESTS:
            time.sleep(SLEEP_BETWEEN_REQUESTS)

    return items[:max_items]


def fetch_tender_detail(tender_stub):
    tender_id = tender_stub.get("id")
    if not tender_id:
        return None

    r = session.get(f"{BASE_URL}/tenders/{tender_id}", timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    payload = r.json()
    return payload.get("data") if isinstance(payload, dict) else None


# =========================
# Cell 4. Text processing
# =========================

def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        text = "" if text is None else str(text)

    text = text.lower()

    # money
    text = re.sub(r'(?i)\b(?:uah|грн|гривн[іяі]|₴)\s*\d+(?:[ \u00a0]?\d{3})*(?:[.,]\d+)?', ' <MONEY> ', text)
    text = re.sub(r'(?i)\b\d+(?:[ \u00a0]?\d{3})*(?:[.,]\d+)?\s*(?:uah|грн|гривн[іяі]|₴)\b', ' <MONEY> ', text)

    # dates
    text = re.sub(r'\b\d{4}-\d{2}-\d{2}\b', ' <DATE> ', text)
    text = re.sub(r'\b\d{1,2}[./-]\d{1,2}[./-]\d{2,4}\b', ' <DATE> ', text)
    text = re.sub(
        r'\b\d{1,2}\s+(?:січня|лютого|березня|квітня|травня|червня|липня|серпня|вересня|жовтня|листопада|грудня)\s+\d{4}\b',
        ' <DATE> ',
        text
    )

    # urls, emails, hashes, ids
    text = re.sub(r'https?://\S+|www\.\S+', ' <URL> ', text)
    text = re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', ' <EMAIL> ', text)
    text = re.sub(r'\b[0-9a-f]{8,}\b', ' <ID> ', text)

    # numbers
    text = re.sub(r'\b\d+(?:[.,]\d+)?\b', ' <NUM> ', text)

    # non-letter cleanup
    text = re.sub(r'[^a-zа-яіїєґ0-9<>\s\-]+', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def extract_label_from_cpv(cpv: str):
    if not cpv:
        return None

    cpv = re.sub(r"\D", "", str(cpv))

    if len(cpv) < 2:
        return None

    prefix = cpv[:2]

    mapping = {
        # construction / repair
        "45": "construction",
        "50": "repair",

        # medicine / pharma
        "33": "medicine",

        # IT / telecom
        "48": "it",
        "72": "it",
        "32": "electronics",

        # office / education
        "30": "office",

        # food
        "15": "food",
        "03": "agriculture",

        # transport
        "34": "transport",

        # energy / utilities
        "09": "energy",

        # chemicals
        "24": "chemicals",

        # industrial equipment
        "42": "industrial",
        "43": "industrial",

        # services
        "79": "business_services",
        "90": "public_services",
        "98": "public_services",
    }

    return mapping.get(prefix, "other")


def join_text_fields(tender: dict) -> str:
    parts = []

    for key in ["title", "description"]:
        val = tender.get(key)
        if val:
            parts.append(str(val))

    items = tender.get("items") or []
    for item in items:
        if not isinstance(item, dict):
            continue

        if item.get("description"):
            parts.append(str(item["description"]))

        cls = item.get("classification") or {}
        if isinstance(cls, dict):
            if cls.get("id"):
                parts.append(str(cls["id"]))
            if cls.get("scheme"):
                parts.append(str(cls["scheme"]))
            if cls.get("description"):
                parts.append(str(cls["description"]))

        add_cl = item.get("additionalClassifications") or []
        if isinstance(add_cl, list):
            for ac in add_cl:
                if isinstance(ac, dict):
                    if ac.get("id"):
                        parts.append(str(ac["id"]))
                    if ac.get("scheme"):
                        parts.append(str(ac["scheme"]))

    return " ".join(parts)


# =========================
# Cell 5. Build dataset
# =========================

raw_tenders = collect_feed_items(max_items=MAX_FEED_ITEMS)
print("Feed items collected:", len(raw_tenders))

rows = []

for stub in tqdm(raw_tenders, desc="Fetching tender details"):
    try:
        detail = fetch_tender_detail(stub)
    except Exception:
        continue

    if not detail:
        continue

    items = detail.get("items") or []
    if not items:
        continue

    cpv = None
    label = None

    for item in items:
        if not isinstance(item, dict):
            continue
        cls = item.get("classification") or {}
        if isinstance(cls, dict) and cls.get("id"):
            cpv = str(cls["id"])
            label = extract_label_from_cpv(cpv)
            if label is not None:
                break

    if label is None:
        continue

    text_raw = join_text_fields(detail)
    if not text_raw.strip():
        continue

    rows.append({
        "id": detail.get("id"),
        "tenderID": detail.get("tenderID"),
        "dateModified": detail.get("dateModified"),
        "cpv": cpv,
        "label": label,
        "text_raw": text_raw
    })

    if SLEEP_BETWEEN_REQUESTS:
        time.sleep(SLEEP_BETWEEN_REQUESTS)

df = pd.DataFrame(rows)

if df.empty:
    raise ValueError(
        "dataset is empty. Try increasing MAX_FEED_ITEMS or check BASE_URL."
    )

print(df.head())
print("\nLabel distribution:")
print(df["label"].value_counts())


# =========================
# Cell 6. Preprocess text
# =========================

df["text"] = df["text_raw"].apply(normalize_text)
df = df[df["text"].str.len() > 0].copy()

print("\nAfter normalization:")
print(df[["label", "text"]].head())


# =========================
# Cell 7. Train/test split
# =========================

X = df["text"].values
y = df["label"].values


label_counts = df["label"].value_counts()
can_stratify = (label_counts.min() >= 2) and (df["label"].nunique() > 1)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y if can_stratify else None
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))


# =========================
# Cell 8. Models
# =========================

def build_logreg_pipeline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.95,
            max_features=50000,
            sublinear_tf=True,
            strip_accents="unicode",
            stop_words=None
        )),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            n_jobs=None
        ))
    ])


def build_linearsvc_pipeline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.95,
            max_features=50000,
            sublinear_tf=True,
            strip_accents="unicode",
            stop_words=None
        )),
        ("clf", LinearSVC(class_weight="balanced"))
    ])


models = {
    "logreg": build_logreg_pipeline(),
    "linearsvc": build_linearsvc_pipeline(),
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    f1m = f1_score(y_test, preds, average="macro")

    results[name] = {
        "accuracy": acc,
        "macro_f1": f1m,
        "preds": preds,
        "model": model
    }

    print("=" * 80)
    print(name)
    print("accuracy:", round(acc, 4))
    print("macro_f1 :", round(f1m, 4))
    print(classification_report(y_test, preds, zero_division=0))


# =========================
# Cell 9. Pick best model
# =========================

best_name = max(results, key=lambda k: results[k]["macro_f1"])
best_model = results[best_name]["model"]
best_preds = results[best_name]["preds"]

print("\nBest model:", best_name)
print("Accuracy:", round(results[best_name]["accuracy"], 4))
print("Macro F1 :", round(results[best_name]["macro_f1"], 4))


# =========================
# Cell 10. Confusion matrix
# =========================

labels_order = sorted(df["label"].unique())
cm = confusion_matrix(y_test, best_preds, labels=labels_order)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation="nearest")
ax.set_title(f"Confusion Matrix — {best_name}")
ax.set_xticks(range(len(labels_order)))
ax.set_yticks(range(len(labels_order)))
ax.set_xticklabels(labels_order, rotation=45, ha="right")
ax.set_yticklabels(labels_order)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")

fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


# =========================
# Cell 11. Error analysis
# =========================

test_df = pd.DataFrame({
    "text": X_test,
    "true": y_test,
    "pred": best_preds
})

errors = test_df[test_df["true"] != test_df["pred"]].copy()
print("\nMisclassified examples:", len(errors))
display(errors.head(20))

print("\nMost common true/pred pairs among errors:")
if not errors.empty:
    pair_counts = (
        errors.groupby(["true", "pred"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    display(pair_counts.head(20))


# =========================
# Cell 12. Predict on new text
# =========================

def predict_category(text: str):
    cleaned = normalize_text(text)
    pred = best_model.predict([cleaned])[0]
    return pred

sample_text = """
Будівництво капітального ремонту будівлі школи з поставкою матеріалів
на суму 1200000 грн до 2026-09-01.
"""

print("Sample prediction:", predict_category(sample_text))


# =========================
# Cell 13. Save dataset and model summary
# =========================

df.to_csv("prozorro_classic_nlp_dataset.csv", index=False)
print("Saved dataset to prozorro_classic_nlp_dataset.csv")

summary = {
    "rows": int(len(df)),
    "labels": df["label"].value_counts().to_dict(),
    "best_model": best_name,
    "accuracy": float(results[best_name]["accuracy"]),
    "macro_f1": float(results[best_name]["macro_f1"]),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))